# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/susheel123-sketch/Flyrank-Internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring. **What this turns into a playbook:** the ranker that survived my Week-6 audit —
**Logistic Regression on strict-timeline features** (only data from before the label window), scored with a **client-grouped** split.
**Data:** starter slice, 30,000 pages, 32 clients. **Label (evaluation only):** impressions fell >20% between two recent 30-day windows.
**Everything here is decision-support for review order** — not a forecast, and not proof that any action works. **Seed:** 42.

In [1]:
import os, sys, json, subprocess, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
IN_COLAB = "google.colab" in sys.modules
REPO_URL, REPO_DIR = "https://github.com/susheel123-sketch/Flyrank-Internship-ml", "Flyrank-Internship-ml"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
SEED = 42
os.makedirs("work/outputs", exist_ok=True); os.makedirs("work/figures", exist_ok=True)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
y = (df["trend_direction"] == "down").astype(int).values
groups = df["client_id"].values
CAT = ["competition_level", "content_type", "main_intent"]

# Days 61-90 back from export = strictly before the label window (Week-6 audit, identity asserted there).
w61 = pd.DataFrame({"impr": df["impressions_90d"] - df["impressions_last_30d"] - df["impressions_prev_30d"],
                    "clicks": df["clicks_90d"] - df["clicks_last_30d"] - df["clicks_prev_30d"],
                    "sess": df["sessions_90d"] - df["sessions_last_30d"] - df["sessions_prev_30d"]})
assert (w61 >= 0).all().all()
ctr61 = np.where(w61["impr"] > 0, w61["clicks"] / w61["impr"].replace(0, 1) * 100, 0)

def strict_features(d):
    X = pd.DataFrame(index=d.index)
    for c in ["search_volume", "cpc"]:
        X[c] = np.log1p(d[c]).fillna(0); X["has_" + c] = d[c].notna().astype(int)
    X["competition"] = d["competition"].fillna(0)
    X["word_count"] = d["word_count"].fillna(0); X["has_word_count"] = d["word_count"].notna().astype(int)
    X["age_at_cutoff"] = d["content_age_days"] - 60
    X["impr_61_90"] = np.log1p(w61["impr"]); X["clicks_61_90"] = np.log1p(w61["clicks"]); X["sess_61_90"] = np.log1p(w61["sess"])
    X["ctr_61_90"] = ctr61; X["visible_61_90"] = (w61["impr"] > 0).astype(int)
    return X.join(pd.get_dummies(d[CAT].fillna("unknown"), dtype=int))

X = strict_features(df)
FOLDS = list(GroupKFold(5).split(X, y, groups))
score = np.zeros(len(df)); contrib = pd.DataFrame(0.0, index=df.index, columns=X.columns)
for tr, te in FOLDS:      # every page is scored by a model that never saw its client
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(X.iloc[tr], y[tr])
    score[te] = m.predict_proba(X.iloc[te])[:, 1]
    z = m[0].transform(X.iloc[te]); contrib.iloc[te] = z * m[-1].coef_[0]   # per-page push toward 'declining'

def p_at_k(s, k): return float(y[np.argsort(-s, kind="stable")[:k]].mean())
print(f"base rate {y.mean():.3f} | pooled AUC {roc_auc_score(y, score):.3f} | P@50 {p_at_k(score, 50):.2f} | P@200 {p_at_k(score, 200):.2f}")

base rate 0.542 | pooled AUC 0.661 | P@50 0.86 | P@200 0.85


## 1. Ranked actions + reason codes

**How the queue is built:**
1. **Score:** every page gets a decline-risk score from the audited ranker, using a model that never saw that page's client.
2. **Archetype:** a short, readable rule puts each page in one of six groups, and each group has one default action.
3. **Reason code:** `archetype | top driver`. The top driver is the feature that pushed this page's score up most.
4. **Per-client cap:** at most 10 pages per client in the first 50, and 40 in the first 200 (why: see 1c).
5. **Review tiers:** top 50 = this week, next 150 = this month, everyone else = monitor. Near-invisible pages never enter a review tier.

**Why rule-based archetypes, not k-means?** A reviewer has to understand the group in one line. The paper's own k-means clusters ended up with duplicate names. Rules I can state are easier to trust and easier to change.

In [2]:
# 1a. Archetypes: transparent rules (first match wins), built ONLY from pre-label-window data + page metadata.
#     avg_position is a 90-day average (overlaps the label window) -> used only to NAME the low-CTR archetype, never to score.
band = pd.cut(df["avg_position"].where(df["avg_position"] > 0), [0, 3, 10, 20], labels=["1-3", "4-10", "11-20"])
ok = (w61["impr"] >= 300) & band.notna()
band_ctr = pd.Series(ctr61[ok.values], index=df.index[ok]).groupby(band[ok], observed=True).median()
low_ctr = ok & (pd.Series(ctr61, index=df.index) < 0.5 * band.map(band_ctr).astype(float))

rules = [
    ("quiet_page",           w61["impr"] < 100,                                    "no_action_batch_monitor"),
    ("zero_click_visible",   (w61["impr"] >= 300) & (w61["clicks"] == 0),          "verify_tracking_then_snippet_review"),
    ("low_ctr_page_1_2",     low_ctr,                                              "rewrite_title_meta"),
    ("large_established",    w61["impr"] >= 10000,                                 "protect_light_review"),
    ("young_fading_risk",    df["content_age_days"] - 60 < 180,                    "content_refresh_review"),
    ("steady_mid",           pd.Series(True, index=df.index),                      "monitor"),
]
arch = pd.Series(pd.NA, index=df.index, dtype="object"); action = arch.copy()
for name, cond, act in rules:
    hit = arch.isna() & cond
    arch[hit] = name; action[hit] = act

q = pd.DataFrame({"content_id": df["content_id"], "client_id": df["client_id"], "score": score.round(4),
                  "archetype": arch, "action": action, "impr_61_90": w61["impr"], "clicks_61_90": w61["clicks"],
                  "ctr_61_90": np.round(ctr61, 3), "avg_position_90d": df["avg_position"], "content_age_days": df["content_age_days"],
                  "content_type": df["content_type"], "main_intent": df["main_intent"]})
q["declined"] = y   # evaluation only; dropped before export

# 1b. Reason codes: archetype + the feature that pushed this page's score up most (from its own fold's model).
pretty = {"impr_61_90": "earlier_impressions_high", "clicks_61_90": "earlier_clicks_low", "ctr_61_90": "earlier_ctr_low",
          "age_at_cutoff": "page_is_young", "sess_61_90": "earlier_sessions", "visible_61_90": "was_visible",
          "word_count": "word_count", "search_volume": "keyword_volume", "competition": "keyword_competition"}
top_feat = contrib.idxmax(axis=1)
q["reason_code"] = q["archetype"] + "|" + top_feat.map(lambda f: pretty.get(f, f))

# Human-gate flags (Section 3 explains each)
q["flag_check_tracking"] = (q["clicks_61_90"] == 0) & (q["impr_61_90"] >= 300)
q["flag_tiny_label_risk"] = df["impressions_prev_30d"] < 20
q["flag_updated_recently"] = df["days_since_last_update"] <= 60

q = q.sort_values(["score", "impr_61_90"], ascending=False).reset_index(drop=True)
q["model_rank"] = np.arange(1, len(q) + 1)

# Per-client cap: the raw model order is dominated by one client (shown in 1c). A reviewer queue that is 80%+ one site
# is really one site-level problem, so cap slots: max 10 pages per client in the first 50, max 40 in the first 200.
# Quiet pages never enter a review tier.
taken, order = {}, []
for i in q.index[q["archetype"] != "quiet_page"]:
    c = q.at[i, "client_id"]; n = taken.get(c, 0); pos = len(order)
    cap = 10 if pos < 50 else 40
    if pos < 200 and n >= cap:
        continue
    taken[c] = n + 1; order.append(i)
    if len(order) == 200:
        break
q["rank"] = len(q) + 1
q.loc[order, "rank"] = np.arange(1, len(order) + 1)
rest = q.index.difference(order)
q.loc[rest, "rank"] = np.arange(len(order) + 1, len(order) + 1 + len(rest))   # rest keep model order
q = q.sort_values("rank").reset_index(drop=True)
q["review_tier"] = np.select([q["rank"] <= 50, q["rank"] <= 200], ["1_this_week", "2_this_month"], "3_monitor")

# Value lane: precision is not value. Expected clicks at risk per 30 days = score x earlier-window clicks / 1.
q["clicks_at_risk"] = (q["score"] * q["clicks_61_90"]).round(2)

# Archetype -> action map with evidence
amap = q.groupby("archetype").agg(n=("score", "size"), action=("action", "first"), mean_score=("score", "mean"),
                                  decline_rate=("declined", "mean"))
amap["in_top_200"] = q[q["rank"] <= 200]["archetype"].value_counts().reindex(amap.index).fillna(0).astype(int)
display(amap.sort_values("mean_score", ascending=False).round(3))
print("base rate", round(y.mean(), 3))

,n,action,mean_score,decline_rate,in_top_200
archetype,,,,,
zero_click_visible,3809,verify_tracking_then_snippet_review,0.728,0.680,157
low_ctr_page_1_2,1209,rewrite_title_meta,0.709,0.672,22
young_fading_risk,6971,content_refresh_review,0.680,0.675,10
large_established,1150,protect_light_review,0.574,0.557,9
steady_mid,6140,monitor,0.551,0.495,2
quiet_page,10721,no_action_batch_monitor,0.442,0.418,0


base rate 0.542


In [3]:
# 1c. The ranked actions a reviewer would actually see (top 15).
show = ["rank", "model_rank", "content_id", "review_tier", "action", "reason_code", "score", "impr_61_90", "clicks_61_90",
        "avg_position_90d", "content_age_days", "flag_check_tracking", "flag_updated_recently"]
display(q.head(15)[show])
raw200 = q[q["model_rank"] <= 200]
print(f"RAW model order, top 200: largest client share {raw200['client_id'].value_counts(normalize=True).iloc[0]:.0%}, "
      f"{raw200['client_id'].nunique()} clients, decline rate {raw200['declined'].mean():.3f}")
print("CAPPED queue (what the playbook uses) — precision by tier (evaluation only):")
display(q.groupby("review_tier").agg(pages=("rank", "size"), decline_rate=("declined", "mean"),
                                     clients=("client_id", "nunique"), earlier_clicks=("clicks_61_90", "sum")).round(3))
vl = q.nlargest(50, "clicks_at_risk")
print(f"VALUE LANE (top 50 by score x earlier clicks): decline rate {vl['declined'].mean():.3f}, "
      f"earlier-window clicks covered {int(vl['clicks_61_90'].sum())}, clients {vl['client_id'].nunique()}")

,rank,model_rank,content_id,review_tier,action,reason_code,score,impr_61_90,clicks_61_90,avg_position_90d,content_age_days,flag_check_tracking,flag_updated_recently
0,1,1,content_39b6f6812582,1_this_week,verify_tracking_then_snippet_review,zero_click_visible|earlier_impressions_high,0.9375,11145,0,41.4,111,True,True
1,2,2,content_d4786a3f7544,1_this_week,verify_tracking_then_snippet_review,zero_click_visible|earlier_impressions_high,0.9340,3589,0,21.9,229,True,False
2,3,3,content_033d8356c2d9,1_this_week,rewrite_title_meta,low_ctr_page_1_2|earlier_impressions_high,0.9338,22828,1,6.0,118,False,True
3,4,4,content_b2ec40b9d68c,1_this_week,verify_tracking_then_snippet_review,zero_click_visible|earlier_impressions_high,0.9312,5654,0,47.3,139,True,False
4,5,5,content_9966666373a0,1_this_week,verify_tracking_then_snippet_review,zero_click_visible|earlier_impressions_high,0.9264,1764,0,31.3,98,True,True
5,6,6,content_db5d88b3f5d4,1_this_week,verify_tracking_then_snippet_review,zero_click_visible|earlier_impressions_high,0.9247,4681,0,3.6,118,True,True
6,7,7,content_e454bc31017a,1_this_week,verify_tracking_then_snippet_review,zero_click_visible|competition_level_unknown,0.9220,801,0,27.6,229,True,False
7,8,8,content_7468618ed04d,1_this_week,verify_tracking_then_snippet_review,zero_click_visible|earlier_impressions_high,0.9209,9622,0,42.1,111,True,True
8,9,9,content_c6999f7eb5fb,1_this_week,verify_tracking_then_snippet_review,zero_click_visible|earlier_impressions_high,0.9206,16512,0,8.8,133,True,True
9,10,10,content_ec5e5f49929b,1_this_week,verify_tracking_then_snippet_review,zero_click_visible|earlier_impressions_high,0.9190,4140,0,51.3,229,True,False


RAW model order, top 200: largest client share 84%, 6 clients, decline rate 0.850
CAPPED queue (what the playbook uses) — precision by tier (evaluation only):


,pages,decline_rate,clients,earlier_clicks
review_tier,,,,
1_this_week,50,0.86,8,10
2_this_month,150,0.82,10,64
3_monitor,29800,0.54,32,171777


VALUE LANE (top 50 by score x earlier clicks): decline rate 0.480, earlier-window clicks covered 21167, clients 6


**Archetype → action map** (decline rates are measured, out-of-fold; base rate 0.54):

| Archetype | Rule (first match wins) | Default action | Decline rate | Read it as |
|---|---|---|---|---|
| `zero_click_visible` | ≥300 earlier impressions, 0 clicks | verify tracking, then snippet review | 0.68 | most of the top of the queue — check data before work |
| `low_ctr_page_1_2` | positions 1–20, CTR < half its band | rewrite title/meta | 0.67 | the Week-4 signal, kept |
| `young_fading_risk` | page < ~240 days old | content refresh review | 0.68 | early-life fade (see 1d) |
| `large_established` | ≥10k earlier impressions | protect, light review only | 0.56 | near the base rate — don't over-touch big pages |
| `steady_mid` | everything else | monitor | 0.50 | no action |
| `quiet_page` | <100 earlier impressions | no action, batch monitor | 0.42 | too small to read a trend |

**What the queue shows:**
- **The cap was needed, and it costs nothing.** In raw model order, **one client holds 84% of the top 200**. With the cap, the largest share is 20% across 10 clients, and precision in the top 50 stays the same (0.86 vs 0.84 raw).
- **82% of the top 50 are zero-click pages.** That is one pattern, and it might be a tracking gap. So the first action on those pages is a data check, not an edit.
- **Precision is not value (the most important line in this playbook).** The top 50 covers only **10 earlier-window clicks** in total. These are pages losing *impressions* that were barely getting clicks.
  A click-weighted "value lane" (score × earlier clicks) covers **21,167 clicks**, but only 48% of those pages declined, which is **below** the base rate. The ranker is good at finding decline. That is not the same as finding the decline that costs the most.

In [4]:
# 1d. The decay / refresh insight: decline rate by page age — pooled AND inside clients.
age_band = pd.cut(df["content_age_days"], [89, 180, 270, 365, 9999], labels=["90-180d", "181-270d", "271-365d", "365d+"])
pooled = pd.DataFrame({"n": pd.Series(y).groupby(age_band.values).size(), "decline_rate": pd.Series(y).groupby(age_band.values).mean()}).round(3)
young = np.where(df["content_age_days"] <= 270, "young", "old")
within = (pd.DataFrame({"c": groups, "a": young, "y": y}).groupby(["c", "a"])["y"].agg(["size", "mean"])
          .query("size >= 30")["mean"].unstack())
diff = (within["young"] - within["old"]).dropna()
display(pooled)
print(f"Inside clients with >=30 pages both <=270d and >270d old (n={len(diff)}): young-minus-old decline gap median {diff.median():+.3f}, "
      f"young higher in {(diff > 0).sum()} of {len(diff)}")
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.bar(pooled.index.astype(str), pooled["decline_rate"], color="#4C6A92")
ax.axhline(y.mean(), ls="--", c="grey", lw=1); ax.text(3.45, y.mean() + 0.01, "base rate", ha="right", fontsize=8, color="grey")
for i, (r, n) in enumerate(zip(pooled["decline_rate"], pooled["n"])):
    ax.text(i, r + 0.01, f"{r:.0%}\nn={n:,}", ha="center", fontsize=8)
ax.set_ylim(0, 0.8); ax.set_ylabel("share of pages declining"); ax.set_title("Decline rate by page age — pooled across clients (one snapshot)", fontsize=10)
ax.spines[["top", "right"]].set_visible(False); fig.tight_layout(); fig.savefig("work/figures/w07_decline_by_age.png", dpi=150); plt.show()

,n,decline_rate
90-180d,12272,0.627
181-270d,4255,0.590
271-365d,7113,0.470
365d+,6360,0.426


Inside clients with >=30 pages both <=270d and >270d old (n=11): young-minus-old decline gap median +0.017, young higher in 7 of 11


**The decay / refresh insight — stated at the strength the data allows:**
- **Pooled:** younger pages decline more often (63% at 90–180 days vs 43% at 365+). This is the reverse of a late-life "decay cliff".
- **Inside clients:** the gap almost disappears (median +0.02 across 11 clients, young higher in 7 of 11). Most of the pooled pattern is **which clients have young pages**, not age itself.
- **The refresh side:** 69% of pages were updated in the last 60 days, often in client-wide batches (Week 6). That makes "refreshed vs not" mostly a client comparison too.
- **Safe wording for the paper:** "In this snapshot we **observed** more decline among younger pages, but the pattern is **largely between clients**. We did not find evidence for a late-life decay point, and this data cannot measure whether a refresh changes outcomes."
  This sits alongside the FlyRank paper's age curve rather than against it: different metric (health score vs impression trend), different population, and no matched before/after on either side.

## 2. Intended use and limits

**Intended use:**
- **Who:** a content reviewer or SEO lead with a few hours a week.
- **What for:** deciding **which pages to look at first**. The queue is a reading order, not a to-do list.
- **The decision stays human:** the reviewer opens the page, checks the gates in section 3, and decides whether any action is warranted.

**What the evidence supports (claim ladder):**
- **Validated ranking:** on held-out clients, the top 50 has 0.86 declining vs 0.54 at random and 0.70 for the hand-written rule (figure below). That is a **measured, decision-support** result.
- **Associations, not effects:** low CTR, zero clicks and young age are **associated with** decline in this snapshot.
- **Not supported:** that any action *prevents* decline. No action was tested. No one in this data was randomly refreshed.

**Where it stops being valid:**
- **One 90-day snapshot.** The label is "impressions fell between two recent 30-day windows" — a same-snapshot proxy, not a forecast of next month.
- **Starter slice only:** 32 clients, pages ≥90 days old with ≥1 impression. Not tested on the warehouse, on new pages, or on new clients.
- **Clients with extreme base rates** (7 of 32 are below 20% or above 80% declining). There, decline looks site-wide, and page ranking adds little.
- **Value is not modelled.** The label counts a page losing 50 impressions the same as one losing 50,000. Use the value lane for business priority.

In [5]:
# What the queue is good for, in numbers: precision at the review depths a team can actually work.
base_rule = np.zeros(len(df))   # Week-4 rule on the strict timeline (the bar from Week 6), for the comparison figure
imp = w61["impr"] * 3
for tr, te in FOLDS:
    tr_idx, te_idx = df.index[tr], df.index[te]
    b = pd.cut(df["avg_position"].where(df["avg_position"] > 0), [0, 3, 10, 20], labels=["1-3", "4-10", "11-20"])
    okk = (imp[tr_idx] >= 500) & b[tr_idx].notna()
    med = pd.Series(ctr61, index=df.index)[tr_idx][okk].groupby(b[tr_idx][okk], observed=True).median()
    e = b[te_idx].map(med).astype(float); c = pd.Series(ctr61, index=df.index)[te_idx]
    flag = (imp[te_idx] >= 500) & b[te_idx].notna() & (c < 0.5 * e)
    base_rule[te] = np.where(flag, (1 - c / e).clip(0, 1) * np.log1p(imp[te_idx]), 0) + np.log1p(imp[te_idx].values) * 1e-6
Ks = [10, 20, 50, 100, 200, 500, 1000]
curve = pd.DataFrame({"playbook ranker (LR, strict)": [p_at_k(score, k) for k in Ks],
                      "Week-4 rule (strict)": [p_at_k(base_rule, k) for k in Ks],
                      "random order (base rate)": [y.mean()] * len(Ks)}, index=Ks).round(3)
display(curve)
fig, ax = plt.subplots(figsize=(6, 3.4))
for col, st in zip(curve.columns, ["-o", "-s", "--"]):
    ax.plot(curve.index, curve[col], st, label=col, ms=4)
ax.set_xscale("log"); ax.set_xlabel("pages reviewed (top K, log scale)"); ax.set_ylabel("share that declined")
ax.set_ylim(0.4, 1.0); ax.legend(fontsize=8, frameon=False); ax.set_title("Precision@K, client-grouped, out-of-fold", fontsize=10)
ax.spines[["top", "right"]].set_visible(False); fig.tight_layout(); fig.savefig("work/figures/w07_precision_at_k.png", dpi=150); plt.show()

# Where the model is weaker: inside clients with extreme base rates.
cb = pd.Series(y).groupby(groups).mean()
print(f"Clients with base rate <0.2 or >0.8: {((cb < 0.2) | (cb > 0.8)).sum()} of {len(cb)} — inside those, page ranking adds little.")

,"playbook ranker (LR, strict)",Week-4 rule (strict),random order (base rate)
10,0.900,0.700,0.542
20,0.900,0.650,0.542
50,0.860,0.700,0.542
100,0.850,0.750,0.542
200,0.850,0.720,0.542
500,0.792,0.738,0.542
1000,0.770,0.734,0.542


Clients with base rate <0.2 or >0.8: 7 of 32 — inside those, page ranking adds little.


## 3. Human review + the no-go list

**Before acting on any page, a reviewer checks:**
1. **Tracking first.** If the page has 0 clicks on ≥300 impressions (`flag_check_tracking`, 79% of the top 200), confirm clicks are being recorded for that URL. If they aren't, it's a data ticket, not a content task.
2. **Already worked on?** If the page was updated in the last 60 days (`flag_updated_recently`, 69% of the top 200), don't refresh it again. Find out what changed and wait for that change to show up.
3. **Is the "decline" real?** Open the page's impressions over time. A few hundred impressions moving around is noise (`flag_tiny_label_risk`, rare in the top tiers by design).
4. **Is it a site-wide event?** If several queue pages from one client dropped together, treat it as **one** client-level case (algorithm update, migration, tracking change), not N page fixes.
5. **Does the action fit the page?** A position-2 page with ~0% CTR may lose clicks to an answer box on the results page — a new title won't fix that. Note it and move on.

**No-go list — never automate these:**
- Rewriting, merging, redirecting, unpublishing or deleting any page because of its score.
- Bulk title/meta changes across an archetype.
- Changing a client's whole site because the model ranked many of its pages.
- Telling a client their content "is declining" or "will decline" from this output alone.
- Using the score for anything about people (writers, teams, vendors) — it was never built for that.
- Using `large_established` pages as targets for heavy rewrites. Their decline rate is near the base rate, and they carry most of the clicks.

**Safe to automate:** regenerating the queue, computing the flags, and the monitoring checks in section 4.

In [6]:
# How often each human gate fires where it matters (top 200).
top200 = q[q["rank"] <= 200]
gates = pd.DataFrame({
    "gate": ["check tracking (0 clicks on >=300 impressions)", "tiny-label risk (<20 earlier impressions)",
             "already updated in last 60 days", "one client holds >25% of the tier"],
    "share_of_top_200": [top200["flag_check_tracking"].mean(), top200["flag_tiny_label_risk"].mean(),
                         top200["flag_updated_recently"].mean(),
                         float(top200["client_id"].value_counts(normalize=True).iloc[0] > 0.25)],
    "decline_rate_when_flagged": [top200.loc[top200["flag_check_tracking"], "declined"].mean(),
                                  top200.loc[top200["flag_tiny_label_risk"], "declined"].mean(),
                                  top200.loc[top200["flag_updated_recently"], "declined"].mean(), np.nan]}).round(3)
display(gates)
print("Largest single-client share of the top 200:", f"{top200['client_id'].value_counts(normalize=True).iloc[0]:.0%}",
      "| clients in top 200:", top200["client_id"].nunique())

,gate,share_of_top_200,decline_rate_when_flagged
0,check tracking (0 clicks on >=300 impressions),0.785,0.809
1,tiny-label risk (<20 earlier impressions),0.010,0.500
2,already updated in last 60 days,0.690,0.841
3,one client holds >25% of the tier,0.000,NaN


Largest single-client share of the top 200: 20% | clients in top 200: 10


## 4. Monitoring / retrain triggers

A light, monthly check — one person, one notebook run. The table below lists each trigger, today's reference value, and what to do.

**Two triggers already fire today, and the playbook says so openly:**
- **Client concentration:** raw model order is 83% one client. Handled by the per-client cap. Keep the cap.
- **Zero-click share of the top 50 is 0.82**, above the 0.80 line. The tracking check in section 3 is therefore **mandatory**, not optional, until someone confirms those zeros are real.

In [7]:
# Reference values for monitoring — recorded now so next month has something to compare against.
top50 = q[q["rank"] <= 50]
reference = {
    "base_rate": round(float(y.mean()), 4),
    "precision_at_50": round(p_at_k(score, 50), 4), "precision_at_200": round(p_at_k(score, 200), 4),
    "rule_bar_precision_at_50": round(p_at_k(base_rule, 50), 4),
    "auc": round(float(roc_auc_score(y, score)), 4),
    "zero_click_share_top_50": round(float(top50["flag_check_tracking"].mean()), 4),
    "largest_client_share_top_200": round(float(top200["client_id"].value_counts(normalize=True).iloc[0]), 4),
    "largest_client_share_top_200_raw_model_order": round(float(raw200["client_id"].value_counts(normalize=True).iloc[0]), 4),
    "precision_at_50_capped_queue": round(float(q.head(50)["declined"].mean()), 4),
    "median_impr_61_90": float(w61["impr"].median()),
    "archetype_mix": q["archetype"].value_counts(normalize=True).round(4).to_dict(),
}
triggers = pd.DataFrame([
    ("Queue quality", "P@50 of the capped queue on the newest month", f"{reference['precision_at_50_capped_queue']:.2f}", "below the Week-4 rule bar on the same month (now 0.70) — or below 0.70 two months running", "retrain; if still below, fall back to the rule"),
    ("Label drift", "base rate (share declining)", f"{reference['base_rate']:.2f}", "moves more than ±0.10", "re-check the label and P@K against the new base rate before trusting anything"),
    ("One-pattern risk", "zero-click share of top 50", f"{reference['zero_click_share_top_50']:.2f}", "above 0.80, or tracking issue confirmed on any client", "pause those pages; fix tracking first"),
    ("Client concentration", "largest client share of top 200 (raw model order)", f"{reference['largest_client_share_top_200_raw_model_order']:.2f}", "above 0.40 — ALREADY FIRING today", "keep the per-client cap; review that client's pattern as one site-level case"),
    ("Input drift", "median earlier-window impressions", f"{reference['median_impr_61_90']:.0f}", "moves more than 30%", "check for new clients or a reporting change"),
    ("New clients", "share of pages from clients unseen in training", "0 (all scored out-of-fold)", "any client with <3 months of history", "score with the rule only until history builds"),
    ("Calendar", "time since last retrain", "—", "every quarter, or after any of the above", "retrain on the latest window; re-run the Week-6 audit cells"),
], columns=["what", "metric", "today", "trigger", "what to do"])
display(triggers)

,what,metric,today,trigger,what to do
0,Queue quality,P@50 of the capped queue on the newest month,0.86,below the Week-4 rule bar on the same month (n...,"retrain; if still below, fall back to the rule"
1,Label drift,base rate (share declining),0.54,moves more than ±0.10,re-check the label and P@K against the new bas...
2,One-pattern risk,zero-click share of top 50,0.82,"above 0.80, or tracking issue confirmed on any...",pause those pages; fix tracking first
3,Client concentration,largest client share of top 200 (raw model order),0.83,above 0.40 — ALREADY FIRING today,keep the per-client cap; review that client's ...
4,Input drift,median earlier-window impressions,308,moves more than 30%,check for new clients or a reporting change
5,New clients,share of pages from clients unseen in training,0 (all scored out-of-fold),any client with <3 months of history,score with the rule only until history builds
6,Calendar,time since last retrain,—,"every quarter, or after any of the above",retrain on the latest window; re-run the Week-...


## 5. Exports for the paper

**What gets written:**
- `work/outputs/action_queue.csv` — the full ranked queue. Pseudonymous `content_id` only: no client ID, no label. Git-ignored by design; this notebook regenerates it.
- `work/figures/w07_precision_at_k.png` — ranker vs rule vs random. Committed.
- `work/figures/w07_decline_by_age.png` — the age pattern, labelled as pooled. Committed.
- `work/outputs/w07_playbook_metrics.json` — every number the paper will quote, plus the monitoring reference values. Committed.

The cost/value table below assumes **15 minutes of review per page**. That assumption is mine — change it to the team's real number.

In [8]:
# The queue the paper builds on. Evaluation label and client id are NOT exported (pseudonymous content_id only).
EXPORT = ["rank", "model_rank", "content_id", "review_tier", "action", "archetype", "reason_code", "score",
          "impr_61_90", "clicks_61_90", "ctr_61_90", "clicks_at_risk", "avg_position_90d", "content_age_days", "content_type", "main_intent",
          "flag_check_tracking", "flag_tiny_label_risk", "flag_updated_recently"]
assert "declined" not in EXPORT and "client_id" not in EXPORT
q[EXPORT].to_csv("work/outputs/action_queue.csv", index=False)

cost = {}
for label, sel in [("capped_top_50", q.head(50)), ("capped_top_200", q.head(200)), ("raw_model_top_50", q.nsmallest(50, "model_rank")),
                   ("value_lane_50", q.nlargest(50, "clicks_at_risk")), ("random_50", q.sample(50, random_state=SEED))]:
    cost[label] = {"pages": len(sel), "declining_pages_found": int(sel["declined"].sum()),
                   "review_hours_at_15min": len(sel) * 0.25,
                   "hours_per_declining_page_found": round(len(sel) * 0.25 / max(sel["declined"].sum(), 1), 3),
                   "earlier_window_clicks_covered": int(sel["clicks_61_90"].sum()), "clients": int(sel["client_id"].nunique())}
display(pd.DataFrame(cost).T)

metrics = {"assignment": "ML-10", "seed": SEED, "model": "LogisticRegression on strict-timeline features, GroupKFold(5) by client, out-of-fold",
           "reference_for_monitoring": reference, "precision_curve": curve.to_dict(),
           "archetype_map": amap.round(4).reset_index().to_dict(orient="records"),
           "decay_by_age": pooled.reset_index().rename(columns={"index": "age_band"}).to_dict(orient="records"),
           "cost_value": cost, "assumptions": {"review_minutes_per_page": 15, "value_proxy": "clicks in days 61-90 (not revenue)"},
           "exports": ["work/outputs/action_queue.csv (git-ignored, regenerated)", "work/figures/w07_precision_at_k.png", "work/figures/w07_decline_by_age.png"]}
json.dump(metrics, open("work/outputs/w07_playbook_metrics.json", "w"), indent=2, default=str)
for f in ["work/outputs/action_queue.csv", "work/outputs/w07_playbook_metrics.json", "work/figures/w07_precision_at_k.png", "work/figures/w07_decline_by_age.png"]:
    print(f"{f:45s} {os.path.getsize(f):>9,} bytes")

,pages,declining_pages_found,review_hours_at_15min,hours_per_declining_page_found,earlier_window_clicks_covered,clients
capped_top_50,50.0,43.0,12.5,0.291,10.0,8.0
capped_top_200,200.0,166.0,50.0,0.301,74.0,10.0
raw_model_top_50,50.0,42.0,12.5,0.298,18.0,4.0
value_lane_50,50.0,24.0,12.5,0.521,21167.0,6.0
random_50,50.0,30.0,12.5,0.417,189.0,18.0


work/outputs/action_queue.csv                 5,681,549 bytes
work/outputs/w07_playbook_metrics.json            4,293 bytes
work/figures/w07_precision_at_k.png              54,803 bytes
work/figures/w07_decline_by_age.png              51,993 bytes


**Cost / value, plainly:**
- **Reviewing the capped top 50 takes about 12.5 hours and finds 43 declining pages** — about 17 minutes per real decline. A random 50 finds 30 (25 minutes per decline). The ranker saves review time per decline found.
- **But those 43 pages had only 10 clicks between them** in the earlier window. Finding them is cheap; fixing them may be worth very little.
- **The value lane is the opposite trade:** 21,167 clicks covered, but only 24 of 50 declined (below random). Those are big pages, mostly stable.

**Recommendation for the paper's playbook section:**
- Run **two short lists**, not one:
  1. A **capped risk queue** (top 50) for "what is slipping".
  2. A **value lane** (top 20 by score × clicks) for "what would hurt most if it slipped".
- Every page goes through the section-3 gates first.
- **Next modelling step:** a click-weighted label, so risk and value point at the same pages.

**One-sentence version (safe language):** *"On held-out clients, the ranker put declining pages at the top of a review list more often than a hand-written rule (P@50 0.86 vs 0.70, base rate 0.54). This is decision-support for review order. It does not show that any action prevents decline, and it does not yet weigh how much traffic is at stake."*

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed before commit; seed 42)
- [x] No client names, URLs, or private queries anywhere (export holds pseudonymous content IDs only, no client IDs, no label)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Ranked actions with reason codes, archetype → action map, decay/refresh insight
- [x] Intended use, limits, human-review rules, no-go list, cost/value, monitoring + retrain triggers
- [x] Exports written by the notebook: queue CSV (git-ignored), figures in `work/figures/`, receipts in `work/outputs/w07_playbook_metrics.json`